# 09 - Compare Classification Models

**Phase 2, Step 2.5 - Model Comparison & Selection**

Run this AFTER training all 4 classifiers (notebooks 05-08). Loads each model's `outputs/metrics_<key>.json`, builds comparison charts, and recommends a best model based on an accuracy/speed tradeoff score.

### 1. Mount Drive and load configuration

In [ ]:
!pip install -q datasets pyyaml tqdm scikit-learn matplotlib pandas opencv-python-headless

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# ---------------------------------------------------------------------------
# Project configuration - shared across every SmartVision AI notebook.
# All notebooks read/write under this same Google Drive folder so that
# work done in one notebook (e.g. dataset collection) is available to the
# next one (e.g. training), even across separate Colab sessions.
# ---------------------------------------------------------------------------
BASE_DIR = "/content/drive/MyDrive/SmartVisionAI"

RAW_DATA_DIR = os.path.join(BASE_DIR, "raw_data")
RAW_IMAGES_DIR = os.path.join(RAW_DATA_DIR, "images")
RAW_ANNOTATIONS_PATH = os.path.join(RAW_DATA_DIR, "annotations.json")

CLASSIFICATION_DIR = os.path.join(BASE_DIR, "classification")
CLASSIFICATION_TRAIN_DIR = os.path.join(CLASSIFICATION_DIR, "train")
CLASSIFICATION_VAL_DIR = os.path.join(CLASSIFICATION_DIR, "val")
CLASSIFICATION_TEST_DIR = os.path.join(CLASSIFICATION_DIR, "test")

DETECTION_DIR = os.path.join(BASE_DIR, "detection")
DETECTION_IMAGES_DIR = os.path.join(DETECTION_DIR, "images")
DETECTION_LABELS_DIR = os.path.join(DETECTION_DIR, "labels")
DETECTION_YAML_PATH = os.path.join(DETECTION_DIR, "data.yaml")

MODELS_DIR = os.path.join(BASE_DIR, "models")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")

for d in [BASE_DIR, RAW_DATA_DIR, RAW_IMAGES_DIR, CLASSIFICATION_DIR, DETECTION_DIR, MODELS_DIR, OUTPUTS_DIR]:
    os.makedirs(d, exist_ok=True)

# The 25 selected COCO classes (must match COCO category names exactly)
SELECTED_CLASSES = [
    # Vehicles (6)
    "car", "truck", "bus", "motorcycle", "bicycle", "airplane",
    # Person (1)
    "person",
    # Outdoor (3)
    "traffic light", "stop sign", "bench",
    # Animals (6)
    "dog", "cat", "horse", "bird", "cow", "elephant",
    # Kitchen & food (5)
    "bottle", "cup", "bowl", "pizza", "cake",
    # Furniture & indoor (4)
    "chair", "couch", "bed", "potted plant",
]
assert len(SELECTED_CLASSES) == 25

CLASS_TO_IDX = {name: i for i, name in enumerate(SELECTED_CLASSES)}
IDX_TO_CLASS = {i: name for i, name in enumerate(SELECTED_CLASSES)}

def safe_name(class_name):
    return class_name.replace(" ", "_")

IMAGES_PER_CLASS = 350        # -> 8,750 images total (up from 100/class to fight overfitting)
TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

CLS_IMG_SIZE = 224            # Classification input resolution (single-resolution throughout)
FINE_TUNE_IMG_SIZE = 384      # Unused by classifier training (reverted to single-resolution); kept for compatibility
YOLO_IMG_SIZE = 640
BATCH_SIZE = 32                # Stage 1 batch size
BATCH_SIZE_STAGE2 = 16         # Smaller batch at 384x384 to fit GPU memory (~2.9x pixels/image)

HF_DATASET_NAME = "detection-datasets/coco"

print("BASE_DIR:", BASE_DIR)
print("Classes:", len(SELECTED_CLASSES))


In [ ]:
import json
import glob

MODEL_KEYS = ["vgg16", "resnet50", "mobilenetv2", "efficientnetb0"]

all_metrics = {}
for key in MODEL_KEYS:
    path = os.path.join(OUTPUTS_DIR, f"metrics_{key}.json")
    if os.path.exists(path):
        with open(path) as f:
            all_metrics[key] = json.load(f)
    else:
        print(f"WARNING: {path} not found - train that model first.")

print(f"Loaded metrics for: {list(all_metrics.keys())}")


### 2. Bar chart: accuracy, precision, recall, F1 across models

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

names = [m["model"] for m in all_metrics.values()]
metrics_to_plot = ["test_accuracy", "precision_macro", "recall_macro", "f1_macro"]
x = np.arange(len(names))
width = 0.2

plt.figure(figsize=(10, 6))
for i, metric in enumerate(metrics_to_plot):
    values = [m[metric] for m in all_metrics.values()]
    plt.bar(x + i * width, values, width, label=metric)
plt.xticks(x + width * 1.5, names)
plt.ylabel("Score")
plt.ylim(0, 1.0)
plt.title("Classification model comparison")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "model_comparison_bars.png"), dpi=150)
plt.show()


### 3. Accuracy vs. inference speed tradeoff

In [ ]:
plt.figure(figsize=(8, 6))
for key, m in all_metrics.items():
    plt.scatter(m["avg_inference_ms"], m["test_accuracy"], s=120)
    plt.annotate(m["model"], (m["avg_inference_ms"], m["test_accuracy"]), textcoords="offset points", xytext=(8, 5))
plt.xlabel("Avg inference time (ms/image)")
plt.ylabel("Test accuracy")
plt.title("Accuracy vs. inference speed tradeoff")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "accuracy_vs_speed.png"), dpi=150)
plt.show()


### 4. Confusion matrices

In [ ]:
n = len(all_metrics)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
if n == 1:
    axes = [axes]
for ax, (key, m) in zip(axes, all_metrics.items()):
    cm = np.array(m["confusion_matrix"])
    ax.imshow(cm, cmap="Blues")
    ax.set_title(m["model"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "confusion_matrices.png"), dpi=150)
plt.show()


### 5. Select the best model (weighted accuracy + speed score)

In [ ]:
def select_best(all_metrics, accuracy_weight=0.7, speed_weight=0.3):
    accs = [m["test_accuracy"] for m in all_metrics.values()]
    speeds = [m["avg_inference_ms"] for m in all_metrics.values()]
    acc_min, acc_max = min(accs), max(accs)
    speed_min, speed_max = min(speeds), max(speeds)

    scores = {}
    for key, m in all_metrics.items():
        norm_acc = (m["test_accuracy"] - acc_min) / (acc_max - acc_min + 1e-9)
        norm_speed = 1 - (m["avg_inference_ms"] - speed_min) / (speed_max - speed_min + 1e-9)
        scores[key] = accuracy_weight * norm_acc + speed_weight * norm_speed
    best_key = max(scores, key=scores.get)
    return best_key, scores

best_key, scores = select_best(all_metrics)

summary = {
    "scores": scores,
    "recommended_model": best_key,
    "recommended_model_name": all_metrics[best_key]["model"],
    "reasoning": "Weighted 70% test accuracy / 30% inference speed.",
}
with open(os.path.join(OUTPUTS_DIR, "model_selection_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("Model comparison summary:")
for key, m in all_metrics.items():
    print(f"  {m['model']:<15} acc={m['test_accuracy']:.4f}  speed={m['avg_inference_ms']:.1f}ms  score={scores[key]:.4f}")
print(f"\nRecommended model: {all_metrics[best_key]['model']}")


**Next notebook:** `10_YOLO_Dataset_Preparation.ipynb` builds the detection dataset so we can train YOLOv8 for multi-object localization.